In [2]:
import yaml
import torch
import pandas as pd
import csv

from data.data_loader import load_data, extract_sessions, build_turn_snapshots
from models.model_loader import load_model
from storage.state_io import save_state, load_state, feed_synthetic_ssm_state
from storage.history_text_io import save_text, load_text, concatenate_texts
from benchmarks.latency_memory_bench import measure_baseline_latency, measure_state_management_latency, print_benchmark_results, plot_memory_growth, plot_latency_comparison, get_memory_size_kb

def read_config(config_path: str):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

d:\Hu_Module\Master\Semester 4\Study Project\Linear attention state management\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_turns(session):
    turns = []
    instruction = ""
    user_chat = ""
    assistant_chat = ""
    turn_index = 0
    for turn in session:
        if turn["role"] == "system":
            instruction = "System: " + turn["content"]
        elif turn["role"] == "user":
            user_chat = "User: " + turn["content"]
        elif turn["role"] == "assistant":
            assistant_chat = "Assistant: " + turn["content"]
            turns.append((instruction, user_chat, assistant_chat))
            user_chat = ""
            assistant_chat = ""
    return turns

In [4]:
config = read_config("configs/config1.yaml")
    
paths = config["paths"]
output_dir = paths["output_dir"]
text_history_dir = paths["text_history_dir"]+"/history.txt"
state_dir = paths["state_dir"]+"/state.pt"
plot_dir = paths["plot_dir"]
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
model, tokenizer = load_model(config["model"]["name"], 
                                  device=config["model"]["device"], 
                                  dtype=getattr(torch, config["model"]["dtype"]))
    
dataset = load_data(config["data"]["name"], 
                    split=config["data"]["split"])

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.
Loading weights: 100%|██████████| 242/242 [00:00<00:00, 427.20it/s, Materializing param=backbone.norm_f.weight]                  


In [6]:
sessions = extract_sessions(dataset)
states = []
session = sessions[0]
snapshots = build_turn_snapshots(session)

experiment_2_benchmark_path = output_dir + "/experiment_2_benchmark.csv"

In [27]:
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**inputs, use_cache=True, labels=inputs["input_ids"])
    
    if role == "assistant":
        loss = output.loss.item() if output.loss is not None else 0.0
        ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
        print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss : {loss:.4f}")

Turn 2 - Perplexity: 28.1301 - Loss : 3.3368
Turn 4 - Perplexity: 35.0705 - Loss : 3.5574
Turn 6 - Perplexity: 36.2693 - Loss : 3.5910
Turn 8 - Perplexity: 34.6958 - Loss : 3.5466
Turn 10 - Perplexity: 36.5096 - Loss : 3.5976
Turn 12 - Perplexity: 35.2124 - Loss : 3.5614


In [9]:
from Test import state_utils, evaluate

In [34]:
cumulative_tokens = 0

for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    n_tokens = inputs["input_ids"].size(1)

    if turn_id == 0:
        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                labels=inputs["input_ids"]
            )

        torch.save(output.cache_params, "cache.pt")

        cumulative_tokens = n_tokens

    else:
        prev_cache = torch.load("cache.pt",  weights_only=False)
        cache_position = torch.arange(
            cumulative_tokens,
            cumulative_tokens + n_tokens,
            device=device
        )

        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                labels=inputs["input_ids"],
                cache_params=prev_cache, cache_position=cache_position
            )

        torch.save(output.cache_params, "cache.pt")

        cumulative_tokens += n_tokens

        if role == "assistant":
            loss = output.loss.item() if output.loss is not None else 0.0
            ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
            print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss: {loss:.4f}")

Turn 2 - Perplexity: 18446.4824 - Loss: 9.8226
Turn 4 - Perplexity: 456241.2500 - Loss: 13.0308
Turn 6 - Perplexity: 23546.8359 - Loss: 10.0667
Turn 8 - Perplexity: 33722164.0000 - Loss: 17.3337
Turn 10 - Perplexity: 47395864.0000 - Loss: 17.6740
Turn 12 - Perplexity: 3014269.5000 - Loss: 14.9189


In [37]:
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    n_tokens = inputs["input_ids"].size(1)
    
    if turn_id == 0:
        with torch.no_grad():
            output = model(**inputs, use_cache=True, labels=inputs["input_ids"])
        
        ssm_state = [s[0].detach().cpu() for s in output.cache_params.ssm_states]
        conv_state = [conv[0].detach().cpu() for conv in output.cache_params.conv_states]
        state = {"ssm": ssm_state, "conv": conv_state}
        save_state(
            state,
            "state_original_ssm.pt"
        )
    else:
        prev_original = load_state("state_original_ssm.pt")
    
        cache_position = torch.arange(len(inputs["input_ids"][0]), device=model.device)
        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                cache_position=cache_position,
                labels=inputs["input_ids"],
                cache_params=state_utils.feed_synthetic_state(model, prev_original["ssm"], prev_original["conv"])
            )
        ssm_state = [s[0].detach().cpu() for s in output.cache_params.ssm_states] # [B, ]
        conv_state = [conv[0].detach().cpu() for conv in output.cache_params.conv_states]
        state = {"ssm": ssm_state, "conv": conv_state}
        save_state(
            state,
            "state_original_ssm.pt"
        )

        if role == "assistant":
            loss = output.loss.item() if output.loss is not None else 0.0
            ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
            print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss: {loss:.4f}")

Turn 2 - Perplexity: 46024.5352 - Loss: 10.7369
Turn 4 - Perplexity: 2852013.7500 - Loss: 14.8635
Turn 6 - Perplexity: 1172889010176.0000 - Loss: 27.7905
Turn 8 - Perplexity: 4701696.0000 - Loss: 15.3634
Turn 10 - Perplexity: 326642.9062 - Loss: 12.6966
Turn 12 - Perplexity: 50030420.0000 - Loss: 17.7281
